In [23]:
%load_ext autoreload
%autoreload 2
import warnings
from pandas.errors import SettingWithCopyWarning

warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
from app.logger import *
import json5,json
import fitz #type: ignore

from app.insur.fund_data import *
from app.utils import *
from app.konstant import get_config, get_regex

from app.amc.fund_data import *

utils = Helper()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
#INSURANCE FUND
amc_id = '79_0'
path = r"79_30-Apr-26_IF.pdf"
config = get_config("2026",amc_id)
regex = get_regex("2026")

object = CanaraHSBCLifeINSR(config,regex,path)
title,path_pdf= object.check_and_highlight(path)
# print("done")
data = object.get_data(path_pdf,title)
extracted_text = object.get_generated_content(data)

C:\Users\kaustubh.keny\Projects\OFFICE PROJECTS\rep_fsparse\config\2026\79_0_AMC.json5


In [17]:
title

{3: 'INDIA MULTI-CAP EQUITY FUND',
 4: 'EQUITY II FUND',
 5: 'EMERGING LEADERS EQUITY FUND',
 6: 'BALANCED PLUS FUND',
 8: 'LARGE CAP ADVANTAGE FUND',
 9: 'INDIA MANUFACTURING FUND',
 10: 'EQUITY FUND',
 11: 'LIQUID FUND',
 12: 'DEBT FUND',
 13: 'NextGen Consumption Fund',
 14: 'GROWTH PLUS FUND',
 16: 'DEBT PLUS FUND',
 17: 'BALANCED FUND',
 19: 'BALANCED II FUND',
 21: 'PENSION GROWTH FUND',
 23: 'GROWTH FUND',
 25: 'MIDCAP MOMENTUM GROWTH INDEX FUND',
 26: 'GROWTH II FUND',
 28: 'PENSION BALANCED FUND',
 29: 'MULTICAP MOMENTUM QUALITY INDEX FUND',
 30: 'NIFTY ALPHA 50 INDEX FUND',
 31: 'NIFTY 500 MULTIFACTOR 50 INDEX FUND',
 32: 'PENSION NIFTY ALPHA 50 INDEX FUND',
 33: 'PENSION DEBT FUND',
 34: 'DISCONTINUED POLICY FUND',
 35: 'PENSION DISCONTINUED POLICY FUND'}

In [37]:
# object = GeneraliLifeINSR(config,regex,path)
final_text = object.refine_extracted_data(extracted_text)
dfs = object.merge_and_select_data(final_text)


In [38]:
save_path = os.path.join(object.JSON_PATH, object.FILE_NAME).replace(".pdf", ".json")
with open(save_path, 'w') as f:
  json.dump(dfs, f, indent=2)
  
with open("data.json","w+") as file:
    json.dump(final_text,file)
with open("extract.json","w+") as file:
    json.dump(extracted_text,file)
    
print(f"File Saved At: {save_path}")

File Saved At: C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\79_30-Apr-26_IF.json


In [43]:
pattern = "[\\d\\.\\,]+(?!\\%)"
for fund, content in final_text.items():
    # check = 'before.fund_manager'
    for key in content:
        if key.endswith("related_instruments"):
            print(fund)
            text =re.sub("[^A-Za-z0-9\\s\\-\\(\\)\\.\\,\\+\\%\\:\\&]+", "",content[key]).strip()
            print(text)
            match = re.findall(pattern,text, re.IGNORECASE)
            print(match)

INDIA MULTI-CAP EQUITY FUND
98.9% 5,111.3 ACTUAL
['98.', '5,111.3']
EQUITY II FUND
98.9% 3,340.5 ACTUAL
['98.', '3,340.5']
EMERGING LEADERS EQUITY FUND
98.8% 2,109.2 ACTUAL
['98.', '2,109.2']
BALANCED PLUS FUND

[]
LARGE CAP ADVANTAGE FUND
93.7% 1,153.3 ACTUAL
['93.', '1,153.3']
INDIA MANUFACTURING FUND
97.5% 930.8 ACTUAL
['97.', '930.8']
EQUITY FUND
98.8% 653.9 ACTUAL
['98.', '653.9']
LIQUID FUND
54.3% 600.8 ACTUAL
['54.', '600.8']
DEBT FUND
91.4% 498.7 ACTUAL
['91.', '498.7']
NextGen Consumption Fund
98.2% 483.6 ACTUAL
['98.', '483.6']
GROWTH PLUS FUND

[]
DEBT PLUS FUND
95.0% 255.4 ACTUAL
['95.', '255.4']
BALANCED FUND

[]
BALANCED II FUND

[]
PENSION GROWTH FUND

[]
GROWTH FUND

[]
MIDCAP MOMENTUM GROWTH INDEX FUND
98.7% 115.6 ACTUAL
['98.', '115.6']
GROWTH II FUND

[]
PENSION BALANCED FUND

[]
MULTICAP MOMENTUM QUALITY INDEX FUND
98.9% 59.4 ACTUAL
['98.', '59.4']
NIFTY ALPHA 50 INDEX FUND
97.9% 35.8 ACTUAL
['97.', '35.8']
NIFTY 500 MULTIFACTOR 50 INDEX FUND
98.4% 19.0 ACTUAL
['98.

In [ ]:
import json
import csv
from typing import Union, Dict, Any, List


def json_to_csv(input_data: str,output_file: str,spacing: int = 2) -> int:
    """
    Convert mutual fund JSON into a structured CSV.
    Parameters:
        output_file (str): Path to output CSV file
        spacing (int): Number of empty rows after each mutual fund
    Returns:
        int: Total number of rows written (excluding header)
    Raises:
        ValueError: If input data format is invalid
        IOError: If file operations fail
    """
    try:
        with open(input_data, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception:
        raise

    # Validate
    if "records" not in data or not isinstance(data["records"], list):
        raise ValueError("Invalid JSON structure")

    row_count = 0
    try:
        with open(output_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            # Header
            writer.writerow([
                "page",
                "table",
                "sfin",
                "mutual_fund_name",
                "main_scheme_name",
                "portfolio_data_0",
                "portfolio_data_1",
                "monthly_aum_value",
                "empty_col",
            ])

            # Process each record
            for record in data["records"]:
                value = record.get("value", {})
                mf_name = value.get("mutual_fund_name", "")
                scheme_name = value.get("main_scheme_name", "")
                aum_value = value.get("monthly_aaum_value", "")
                portfolio_list: List[Dict[str, Any]] = value.get("portfolio_data", [])
                sfin = value.get("sfin","")

                # Skip if no portfolio data
                if not isinstance(portfolio_list, list):
                    continue

                for item in portfolio_list:
                    writer.writerow([
                        item.get("page", ""),
                        item.get("table", ""),
                        sfin,
                        mf_name,
                        scheme_name,
                        item.get("0", ""),
                        item.get("1", ""),
                        aum_value,
                        "",
                    ])
                    row_count += 1

                # Add spacing rows
                for _ in range(spacing):
                    writer.writerow([])
        return row_count
    except Exception as e:
        raise IOError(f"Error writing CSV: {e}")

In [4]:
path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\81_30-Apr-26_IF.json"
# path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\78_30-Apr-26_IF.json" #AXA
# path = r"C:\Users\kaustubh.keny\Projects\OUTPUTS\amc_output\json\74_30-Apr-26_IF.json" #Bandhan
from pathlib import Path


_path_ = Path(path)
output_path = _path_.name.replace(".json",".csv")
json_to_csv(path,output_path)

481

In [11]:
import fitz
import pprint

path = r"81_30-Apr-26_IF.pdf"

doc = fitz.open(path)

page = doc[6]
words = page.get_text("words")
pprint.pprint(words)

[(279.1639709472656,
  38.09400177001953,
  311.0721130371094,
  44.73600387573242,
  'Classification',
  0,
  0,
  0),
 (312.506103515625,
  38.09400177001953,
  313.70611572265625,
  44.73600387573242,
  '|',
  0,
  0,
  1),
 (315.2601013183594,
  38.09400177001953,
  333.83013916015625,
  44.73600387573242,
  'Internal',
  0,
  0,
  2),
 (482.9800109863281,
  73.63397979736328,
  496.5460205078125,
  80.2759780883789,
  'SFIN',
  1,
  0,
  0),
 (498.0880126953125,
  73.63397979736328,
  506.9200134277344,
  80.2759780883789,
  'No.',
  1,
  0,
  1),
 (508.5159912109375,
  73.63397979736328,
  599.367919921875,
  80.2759780883789,
  'ULIF010231209FUTUREAPEX133',
  1,
  0,
  2),
 (187.94000244140625,
  97.51398468017578,
  224.0299835205078,
  104.1559829711914,
  'SECURITIES',
  2,
  0,
  0),
 (362.4700012207031,
  97.51398468017578,
  394.72003173828125,
  104.1559829711914,
  'HOLDINGS',
  2,
  1,
  0),
 (463.8999938964844,
  97.51398468017578,
  496.82806396484375,
  104.155982971